# OpenPlaque — PCAT reproducibility lock

This notebook performs one final internal-consistency check before the PCAT prototype is frozen for multi-subject testing.

It uses **one shared circular-PCAT sampling function** for the primary +0.75 mm geometry and for all wall-margin sensitivity runs. Therefore the +0.75 mm sensitivity row must reproduce the primary result exactly.

The accepted RCA centerline, 10–50 mm interval, and lumen-radius profile are held fixed. This is a research prototype, not Caristo FAI-Score.


In [ ]:
# FIRST EXECUTABLE CELL: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch pcat-reproducibility-lock-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy matplotlib pandas
print('Repository and packages ready.')


## 1. Load the frozen RCA coordinate/radius model and source CCTA


In [ ]:

import sys, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy.spatial import cKDTree

sys.path.insert(0, '/content/OpenPlaque/src')
from openplaque.study import OpenPlaqueStudy

ROOT = Path('/content/drive/MyDrive/OpenPlaque')
BASE = ROOT / 'PCAT_RCA_10_50'
OUT = ROOT / 'PCAT_RCA_10_50_Reproducibility_Lock'
OUT.mkdir(parents=True, exist_ok=True)

FAT_LO_HU, FAT_HI_HU = -190.0, -30.0
SEG0, SEG1 = 10.0, 50.0
MARGINS_MM = [0.25, 0.50, 0.75, 1.00, 1.25]
PRIMARY_MARGIN_MM = 0.75

DRIVE_ZIP = ROOT / 'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
if not DRIVE_ZIP.exists():
    raise FileNotFoundError(DRIVE_ZIP)
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)

EXTRACT_ROOT = '/content/full_dicom_pcat_repro_lock'
shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
source_img, ct, _ = study.load_series(7)
ct = np.asarray(ct)
sp_xyz = np.array(source_img.GetSpacing(), float)
sp_zyx = sp_xyz[::-1]
voxel_mm3 = float(np.prod(sp_xyz))

cp = BASE / 'rca_centerline_smoothed_zyx.csv'
rp = BASE / 'pcat_local_radius_profile.csv'
if not cp.exists() or not rp.exists():
    raise FileNotFoundError('Run the accepted RCA 10–50 mm PCAT prototype first.')

cl = pd.read_csv(cp)
rad = pd.read_csv(rp)
need_cl = {'z','y','x','arc_mm'}
need_rad = {'arc_mm','lumen_radius_mm'}
if not need_cl.issubset(cl.columns) or not need_rad.issubset(rad.columns):
    raise ValueError('Frozen centerline/radius inputs do not have the expected columns.')

arc = cl.arc_mm.to_numpy(float)
pts_zyx = cl[['z','y','x']].to_numpy(float)
pts_mm = pts_zyx * sp_zyx
lumen_all = np.interp(arc, rad.arc_mm.to_numpy(float), rad.lumen_radius_mm.to_numpy(float))

segmask = (arc >= SEG0) & (arc <= SEG1)
seg_arc = arc[segmask]
seg_zyx = pts_zyx[segmask]
seg_mm = pts_mm[segmask]
seg_lumen = lumen_all[segmask]

print('CT shape z,y,x:', ct.shape)
print('Spacing x,y,z mm:', tuple(sp_xyz))
print('Frozen RCA segment:', float(seg_arc.min()), 'to', float(seg_arc.max()), 'mm')
print('Segment points:', len(seg_arc))
print('Mean lumen radius:', round(float(seg_lumen.mean()), 3), 'mm')


## 2. Build one immutable voxel map

All circular-wall geometries use exactly the same crop, physical-coordinate nearest-centerline assignment, source CT values, and aorta exclusion. Only the wall margin changes.


In [ ]:

max_outer = float(np.max(seg_lumen + max(MARGINS_MM)))
max_shell_outer = 3.0 * max_outer
pad_mm = max_shell_outer + 3.0

lo = np.floor(np.min(seg_zyx, axis=0) - pad_mm/sp_zyx).astype(int)
hi = np.ceil(np.max(seg_zyx, axis=0) + pad_mm/sp_zyx).astype(int) + 1
lo = np.maximum(lo, 0)
hi = np.minimum(hi, np.array(ct.shape))

crop = ct[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
zz,yy,xx = np.indices(crop.shape)
g_zyx = np.stack([zz+lo[0], yy+lo[1], xx+lo[2]], axis=-1).reshape(-1,3).astype(float)
g_mm = g_zyx * sp_zyx

tree = cKDTree(seg_mm)
dist_mm, nearest_idx = tree.query(g_mm, k=1, workers=-1)
nearest_idx = nearest_idx.astype(int)
nearest_arc = seg_arc[nearest_idx]
nearest_lumen = seg_lumen[nearest_idx]
hu = crop.reshape(-1).astype(float)
fat_hu = (hu >= FAT_LO_HU) & (hu <= FAT_HI_HU)

aorta = None
aorta_candidates = [
    ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',
    ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz',
]
ap = next((p for p in aorta_candidates if p.exists()), None)
if ap is not None:
    ai = sitk.ReadImage(str(ap))
    if ai.GetSize()!=source_img.GetSize() or not np.allclose(ai.GetSpacing(),source_img.GetSpacing()):
        ai = sitk.Resample(ai, source_img, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
    aorta = sitk.GetArrayFromImage(ai) > 0
    aorta_flat = aorta[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]].reshape(-1)
else:
    aorta_flat = np.zeros(len(hu), dtype=bool)

print('Immutable crop z,y,x:', crop.shape)
print('Crop voxels:', len(hu))
print('Aorta exclusion:', ap if ap else 'not available')


## 3. One shared circular-PCAT sampling function


In [ ]:

def compute_circular_pcat(wall_margin_mm):
    """Canonical circular-wall PCAT computation used for BOTH primary and sensitivity runs."""
    margin = float(wall_margin_mm)
    outer = nearest_lumen + margin
    shell_outer = 3.0 * outer  # outward thickness = one local outer diameter = 2*outer radius

    shell = (
        (dist_mm > outer) &
        (dist_mm <= shell_outer) &
        (nearest_arc >= SEG0) &
        (nearest_arc <= SEG1) &
        (~aorta_flat)
    )
    fat = shell & fat_hu
    vals = hu[fat]
    if len(vals) == 0:
        raise RuntimeError(f'No PCAT voxels for margin {margin}')

    radial_out = dist_mm - outer

    radial_rows = []
    for b in np.arange(0.0, 6.0, 0.5):
        fm = fat & (radial_out >= b) & (radial_out < b+0.5)
        vv = hu[fm]
        radial_rows.append({
            'wall_margin_mm': margin,
            'radial_start_mm': float(b),
            'radial_end_mm': float(b+0.5),
            'fat_voxels': int(len(vv)),
            'mean_hu': float(np.mean(vv)) if len(vv) else np.nan,
        })

    long_rows = []
    for b in range(int(SEG0), int(SEG1)):
        fm = fat & (nearest_arc >= b) & (nearest_arc < b+1)
        vv = hu[fm]
        long_rows.append({
            'wall_margin_mm': margin,
            'arc_start_mm': float(b),
            'arc_end_mm': float(b+1),
            'fat_voxels': int(len(vv)),
            'mean_hu': float(np.mean(vv)) if len(vv) else np.nan,
        })

    return {
        'wall_margin_mm': margin,
        'pcat_mean_hu': float(np.mean(vals)),
        'pcat_median_hu': float(np.median(vals)),
        'pcat_sd_hu': float(np.std(vals)),
        'fat_voxels': int(fat.sum()),
        'fat_volume_ml': float(fat.sum()*voxel_mm3/1000.0),
        'shell_voxels': int(shell.sum()),
        'shell_volume_ml': float(shell.sum()*voxel_mm3/1000.0),
        'fat_fraction': float(fat.sum()/max(1,shell.sum())),
        'radial': pd.DataFrame(radial_rows),
        'longitudinal': pd.DataFrame(long_rows),
    }

# PRIMARY and sensitivity are deliberately separate calls to the SAME function.
primary = compute_circular_pcat(PRIMARY_MARGIN_MM)
sens_results = [compute_circular_pcat(m) for m in MARGINS_MM]
sensitivity = pd.DataFrame([{k:v for k,v in r.items() if k not in ('radial','longitudinal')} for r in sens_results])

same = next(r for r in sens_results if abs(r['wall_margin_mm']-PRIMARY_MARGIN_MM) < 1e-12)
mean_diff = float(primary['pcat_mean_hu'] - same['pcat_mean_hu'])
voxel_diff = int(primary['fat_voxels'] - same['fat_voxels'])
shell_diff = int(primary['shell_voxels'] - same['shell_voxels'])

assert mean_diff == 0.0, f'Primary and +0.75 sensitivity means differ: {mean_diff}'
assert voxel_diff == 0, f'Primary and +0.75 fat voxel counts differ: {voxel_diff}'
assert shell_diff == 0, f'Primary and +0.75 shell voxel counts differ: {shell_diff}'

print('PASS: +0.75 mm primary and +0.75 mm sensitivity are bit-identical.')
display(sensitivity)

sensitivity.to_csv(OUT/'pcat_circular_sensitivity_locked.csv', index=False)
pd.DataFrame([{k:v for k,v in primary.items() if k not in ('radial','longitudinal')}]).to_csv(
    OUT/'pcat_canonical_primary.csv', index=False)
primary['radial'].to_csv(OUT/'pcat_canonical_primary_radial.csv', index=False)
primary['longitudinal'].to_csv(OUT/'pcat_canonical_primary_longitudinal.csv', index=False)


## 4. Reconcile with historical prototype outputs and directional sensitivity


In [ ]:

historical_mean = np.nan
historical_file = BASE/'pcat_summary.csv'
if historical_file.exists():
    hist = pd.read_csv(historical_file)
    for c in ['pcat_mean_hu','mean_pcat_hu','openplaque_pcat_mean_hu']:
        if c in hist.columns:
            historical_mean = float(hist.iloc[0][c])
            break

directional_mean = np.nan
directional_summary_candidates = [
    ROOT/'PCAT_RCA_10_50_Directional_OuterWall'/'directional_pcat_summary.csv',
    *ROOT.rglob('directional_pcat_summary.csv')
]
dpath = next((p for p in directional_summary_candidates if p.exists()), None)
if dpath is not None:
    ddf = pd.read_csv(dpath)
    if 'directional_pcat_mean_hu' in ddf.columns:
        directional_mean = float(ddf.iloc[0]['directional_pcat_mean_hu'])

repro = pd.DataFrame([{
    'canonical_primary_margin_mm': PRIMARY_MARGIN_MM,
    'canonical_primary_mean_hu': primary['pcat_mean_hu'],
    'same_function_sensitivity_0p75_mean_hu': same['pcat_mean_hu'],
    'exact_mean_difference_hu': mean_diff,
    'exact_fat_voxel_difference': voxel_diff,
    'exact_shell_voxel_difference': shell_diff,
    'exact_reproducibility_pass': True,
    'historical_prototype_mean_hu': historical_mean,
    'historical_minus_canonical_hu': historical_mean-primary['pcat_mean_hu'] if np.isfinite(historical_mean) else np.nan,
    'directional_interface_mean_hu': directional_mean,
    'directional_minus_canonical_hu': directional_mean-primary['pcat_mean_hu'] if np.isfinite(directional_mean) else np.nan,
}])
repro.to_csv(OUT/'pcat_reproducibility_check.csv', index=False)
display(repro.T)

methods = sensitivity[['wall_margin_mm','pcat_mean_hu','fat_voxels','fat_volume_ml']].copy()
methods['method'] = methods.wall_margin_mm.map(lambda x:f'circular +{x:.2f} mm')
if np.isfinite(directional_mean):
    methods = pd.concat([methods, pd.DataFrame([{
        'wall_margin_mm': np.nan,
        'pcat_mean_hu': directional_mean,
        'fat_voxels': np.nan,
        'fat_volume_ml': np.nan,
        'method':'directional fat-interface'
    }])], ignore_index=True)

methods.to_csv(OUT/'pcat_locked_method_comparison.csv', index=False)


## 5. Locked QC figures and report


In [ ]:

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(sensitivity.wall_margin_mm, sensitivity.pcat_mean_hu, marker='o', label='shared circular sampler')
ax.axhline(primary['pcat_mean_hu'], linestyle='--', label='canonical +0.75 mm')
if np.isfinite(directional_mean):
    ax.axhline(directional_mean, linestyle=':', label='directional interface')
ax.set_xlabel('outer-wall margin (mm)')
ax.set_ylabel('PCAT mean HU')
ax.set_title('Locked PCAT geometry sensitivity')
ax.legend()
plt.tight_layout()
p1 = OUT/'01_locked_geometry_sensitivity.png'
fig.savefig(p1,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(8,5))
lp = primary['longitudinal']
ax.plot((lp.arc_start_mm+lp.arc_end_mm)/2, lp.mean_hu, label='canonical shared sampler +0.75 mm')
dlong_path = ROOT/'PCAT_RCA_10_50_Directional_OuterWall'/'directional_pcat_longitudinal.csv'
if dlong_path.exists():
    dl = pd.read_csv(dlong_path)
    ax.plot((dl.arc_start_mm+dl.arc_end_mm)/2, dl.mean_hu, label='directional interface', alpha=.8)
ax.set_xlabel('arc length from ostium (mm)')
ax.set_ylabel('PCAT mean HU')
ax.set_title('Locked RCA 10–50 mm longitudinal profile')
ax.legend()
plt.tight_layout()
p2 = OUT/'02_locked_longitudinal_profile.png'
fig.savefig(p2,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(8,5))
rp = primary['radial']
ax.plot((rp.radial_start_mm+rp.radial_end_mm)/2, rp.mean_hu, marker='o', label='canonical circular +0.75 mm')
drad_path = ROOT/'PCAT_RCA_10_50_Directional_OuterWall'/'directional_pcat_radial_0p5mm.csv'
if drad_path.exists():
    dr = pd.read_csv(drad_path)
    ax.plot((dr.radial_start_mm+dr.radial_end_mm)/2, dr.mean_hu, marker='o', label='directional interface')
ax.set_xlabel('mm outward from modeled interface')
ax.set_ylabel('PCAT mean HU')
ax.set_title('Radial profile remains exploratory QC')
ax.legend()
plt.tight_layout()
p3 = OUT/'03_locked_radial_qc.png'
fig.savefig(p3,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)

method_means = methods.pcat_mean_hu.dropna().to_numpy(float)
geom_min = float(np.min(method_means))
geom_max = float(np.max(method_means))
geom_max_abs = float(np.max(np.abs(method_means-primary['pcat_mean_hu'])))

report = f"""# OpenPlaque PCAT reproducibility lock

## Canonical prototype definition
- Vessel/segment: RCA, 10–50 mm from ostium
- Adipose attenuation range: {FAT_LO_HU:.0f} to {FAT_HI_HU:.0f} HU
- Canonical circular outer-wall margin: +{PRIMARY_MARGIN_MM:.2f} mm from estimated lumen radius
- Perivascular radial thickness: one local estimated outer diameter
- Aorta exclusion: {'available' if ap is not None else 'not available'}

## Canonical result
- OpenPlaque PCAT mean: **{primary['pcat_mean_hu']:.2f} HU**
- Median: {primary['pcat_median_hu']:.2f} HU
- SD: {primary['pcat_sd_hu']:.2f} HU
- Fat voxels: {primary['fat_voxels']:,}
- Fat volume: {primary['fat_volume_ml']:.3f} mL

## Exact internal reproducibility
The primary +0.75 mm result and the +0.75 mm sensitivity row are now generated by the exact same function and immutable voxel map.

- Mean difference: **{mean_diff:.12f} HU**
- Fat voxel difference: **{voxel_diff}**
- Shell voxel difference: **{shell_diff}**
- PASS: **True**

## Geometry sensitivity
Across the locked circular-margin variants{f' plus the directional interface method' if np.isfinite(directional_mean) else ''}:
- minimum method mean: {geom_min:.2f} HU
- maximum method mean: {geom_max:.2f} HU
- maximum absolute departure from canonical: {geom_max_abs:.2f} HU

The geometry spread is a technical sensitivity range, not a statistical confidence interval.

## Historical implementation
Historical prototype mean: {historical_mean:.2f} HU
Difference historical - canonical: {(historical_mean-primary['pcat_mean_hu']):.2f} HU

The historical value is retained for provenance but is no longer the canonical implementation if it differs from the locked shared sampler.

## Interpretation
Use the overall RCA 10–50 mm PCAT mean as the prototype endpoint. Radial-layer profiles remain exploratory QC until a true outer-wall segmentation is available.
"""
report_path = OUT/'OPENPLAQUE_PCAT_REPRODUCIBILITY_LOCK.md'
report_path.write_text(report)
print(report)


## 6. Package everything for report-back


In [ ]:

files = [
    OUT/'pcat_canonical_primary.csv',
    OUT/'pcat_canonical_primary_radial.csv',
    OUT/'pcat_canonical_primary_longitudinal.csv',
    OUT/'pcat_circular_sensitivity_locked.csv',
    OUT/'pcat_reproducibility_check.csv',
    OUT/'pcat_locked_method_comparison.csv',
    OUT/'OPENPLAQUE_PCAT_REPRODUCIBILITY_LOCK.md',
    OUT/'01_locked_geometry_sensitivity.png',
    OUT/'02_locked_longitudinal_profile.png',
    OUT/'03_locked_radial_qc.png',
]
zip_path = OUT/'OPENPLAQUE_PCAT_REPRODUCIBILITY_LOCK_REPORT_BACK.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        if p.exists():
            zf.write(p, arcname=p.name)

print('Saved:', zip_path)
print()
print('Report-back ZIP search URL:')
print('https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_PCAT_REPRODUCIBILITY_LOCK_REPORT_BACK.zip')
print()
print('Canonical primary CSV search URL:')
print('https://drive.google.com/drive/u/0/search?q=pcat_canonical_primary.csv')
